# This is the v3 accessor to load a netcdf file

The class has got a more generic name instead of "kscale".

The first argument is a list of strings, not yet used!

The second argument is the path to the dataset.

Variable names go into the description.

The description about k-scale data is still there, needs removing.

In [1]:
import pyearthtools.data

from pyearthtools.data import Petdt
from pathlib import Path
from pyearthtools.data.transforms import Transform, TransformCollection
import pyearthtools.pipeline

# import this for use in the variable interrogation
import xarray as xr


In [2]:
pyearthtools.data.indexes.ArchiveIndex

pyearthtools.data.indexes._indexes.ArchiveIndex

In [3]:
class netcdf_file(pyearthtools.data.indexes.ArchiveIndex):


    def __init__(
        self,
        variables: list[str] | str,
        filepath: str,
        *,
        level_value: int | float | list[int | float] | tuple[list | int, ...] | None = None,
        transforms: Transform | TransformCollection | None = None,
    ):
        super().__init__(
            transforms=transforms,
        )
        self.record_initialisation()

        self.filepath=filepath
        self.requested_variables=variables

        # object member
        self.get_variable_names_from_netcdf()

    @property
    def _desc_(self):
        return {
            "singleline": "Met Office k-scale data",
            "range": "20030101",
            "Documentation": "None",
            "Vars": self.variables,
        }

    
    # This is where the path was hardwired, but now it's passed in
    def filesystem(
        self,
        querytime: str | Petdt
    ) -> Path | dict[str, str | Path]:
        return Path(self.filepath)

    def get_variable_names_from_netcdf(self):
        """
        Returns the variable names from a NetCDF file.
    
        Parameters:
            file_path (str): Path to the NetCDF file.
    
        Returns:
            list: A list of variable names in the dataset.
        """
        # Open the NetCDF file using xarray
        ds = xr.open_dataset(self.filepath)
    
        # Extract variable names (data variables)
        self.variables = list(ds.data_vars.keys())
    
        # Close the dataset to free resources
        ds.close()

       # print( variable_names )
    
       # return variable_names


In [4]:
accessor=netcdf_file(['var_I_want'],'/gws/ssde/j25a/mmh_storage/train106/wr3_20030101.nc')

/tmp/ipykernel_7751/3225001281.py:51: FutureWarning: In a future version, xarray will not decode the variable 'forecast_period' into a timedelta64 dtype based on the presence of a timedelta-like 'units' attribute by default. Instead it will rely on the presence of a timedelta64 'dtype' attribute, which is now xarray's default way of encoding timedelta64 values.
To continue decoding into a timedelta64 dtype, either set `decode_timedelta=True` when opening this dataset, or add the attribute `dtype='timedelta64[ns]'` to this variable on disk.
To opt-in to future behavior, set `decode_timedelta=False`.
  ds = xr.open_dataset(self.filepath)


In [5]:
accessor

netcdf_file
	Description                    Met Office k-scale data
		 range                          '20030101'
		 Documentation                  'None'
		 Vars                           ['m01s30i001', 'latitude_longitude', 'm01s30i002', 'potential_t_avg_250m', 'specific_humidity', 'upward_air_velocity', 'level_height_bnds', 'sigma_bnds', 'horizontal_wind_divergence', 'total_precip__rain___snow_']


	Initialisation                 
		 filepath                       '/gws/ssde/j25a/mmh_storage/train106/wr3_20030101.nc'
		 level_value                    None
		 variables                      ['var_I_want']
	Transforms                     
		 StandardCoordinateNames        {'latitude': "['lat', 'Latitude', 'yt_ocean', 'yt']", 'longitude': "['lon', 'Longitude', 'xt_ocean', 'xt']", 'replacement_dictionary': 'None', 'time': "['Time']"}

In [6]:
# The data isn't for this date but it still works
accessor['20020101']

/home/users/train106/PyEarthTools/packages/data/src/pyearthtools/data/indexes/_indexes.py:480: IndexWarning: Could not find time in dataset to select on. Petdt('2002-01-01')
  warnings.warn(


<xarray.Dataset> Size: 409MB
Dimensions:                     (time: 144, latitude: 296, longitude: 343,
                                 bnds: 2)
Coordinates:
  * time                        (time) datetime64[ns] 1kB 2003-01-01T00:10:00...
  * latitude                    (latitude) float32 1kB -19.98 -19.94 ... -8.028
  * longitude                   (longitude) float32 1kB 120.0 120.0 ... 133.9
    forecast_period             (time) timedelta64[ns] 1kB dask.array<chunksize=(144,), meta=np.ndarray>
    forecast_reference_time     datetime64[ns] 8B ...
    level_height                float32 4B ...
    model_level_number          int32 4B ...
    sigma                       float32 4B ...
Dimensions without coordinates: bnds
Data variables:
    m01s30i001                  (time, latitude, longitude) float32 58MB dask.array<chunksize=(144, 296, 343), meta=np.ndarray>
    latitude_longitude          int32 4B ...
    m01s30i002                  (time, latitude, longitude) float32 58MB dask.array<chunksize=(144, 296, 343), meta=np.ndarray>
    potential_t_avg_250m        (time, latitude, longitude) float32 58MB dask.array<chunksize=(144, 296, 343), meta=np.ndarray>
    specific_humidity           (time, latitude, longitude) float32 58MB dask.array<chunksize=(144, 296, 343), meta=np.ndarray>
    upward_air_velocity         (time, latitude, longitude) float32 58MB dask.array<chunksize=(144, 296, 343), meta=np.ndarray>
    level_height_bnds           (bnds) float32 8B dask.array<chunksize=(2,), meta=np.ndarray>
    sigma_bnds                  (bnds) float32 8B dask.array<chunksize=(2,), meta=np.ndarray>
    horizontal_wind_divergence  (time, latitude, longitude) float32 58MB dask.array<chunksize=(144, 296, 343), meta=np.ndarray>
    total_precip__rain___snow_  (time, latitude, longitude) float32 58MB dask.array<chunksize=(144, 296, 343), meta=np.ndarray>
Attributes:
    Conventions:  CF-1.7

In [7]:
pipe1 = pyearthtools.pipeline.Pipeline(
    accessor,
)
pipe1